# Level 3: Retrieval Quality (Hybrid Search)

This notebook compares Dense-Only retrieval (FAISS) against Hybrid Search (FAISS + BM25) across 3 different queries.

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# Initialize models
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

with open('dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

docs = []
for item in dataset:
    doc = Document(page_content=item['content'], metadata={"title": item['title'], "url": item['url']})
    docs.append(doc)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
all_chunks = text_splitter.split_documents(docs)

# Build Dense Retriever (FAISS)
vector_store = FAISS.from_documents(all_chunks, embeddings)
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Build Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(all_chunks)
sparse_retriever.k = 5

C:\Users\Amira Roshdy\AppData\Local\Temp\ipykernel_20868\1427453677.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Implement Reciprocal Rank Fusion (Hybrid)

In [2]:
def hybrid_search(query, k=5):
    dense_results = dense_retriever.invoke(query)
    sparse_results = sparse_retriever.invoke(query)
    
    rrf_scores = {}
    for rank, doc in enumerate(dense_results):
        doc_content = doc.page_content
        if doc_content not in rrf_scores:
            rrf_scores[doc_content] = {"score": 0.0, "doc": doc}
        rrf_scores[doc_content]["score"] += 1.0 / (rank + 60)
            
    for rank, doc in enumerate(sparse_results):
        doc_content = doc.page_content
        if doc_content not in rrf_scores:
            rrf_scores[doc_content] = {"score": 0.0, "doc": doc}
        rrf_scores[doc_content]["score"] += 1.0 / (rank + 60)
        
    sorted_docs = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in sorted_docs][:k]

## Comparison: Dense vs Hybrid Search

In [3]:
test_queries = [
    "What is Azure Chaos Studio?", # Niche/Specific terminology
    "How do I manage relational databases?", # Broad semantic query
    "Error 404 in Azure Web Apps" # Keyword-heavy query
]

for q in test_queries:
    print(f"\n=======================================================")
    print(f"QUERY: {q}")
    print(f"=======================================================\n")
    
    print("--- TOP 3 DENSE RESULTS (FAISS) ---")
    dense_docs = dense_retriever.invoke(q)[:3]
    for i, doc in enumerate(dense_docs):
        print(f"{i+1}. [{doc.metadata['title']}] {doc.page_content[:100]}...")
        
    print("\n--- TOP 3 HYBRID RESULTS (FAISS + BM25) ---")
    hybrid_docs = hybrid_search(q, k=3)
    for i, doc in enumerate(hybrid_docs):
        print(f"{i+1}. [{doc.metadata['title']}] {doc.page_content[:100]}...")
        
    print("\n\n")


QUERY: What is Azure Chaos Studio?

--- TOP 3 DENSE RESULTS (FAISS) ---
1. [Azure for developers overview] that are scalable, reliable, and maintainable. Azure supports the most popular programming languages...
2. [Azure CycleCloud Documentation] Read in English Edit Azure CycleCloud Documentation Azure CycleCloud is designed to enable enterpris...
3. [Azure Database for PostgreSQL documentation] Read in English Edit Azure Database for PostgreSQL documentation Azure Database for PostgreSQL is a ...

--- TOP 3 HYBRID RESULTS (FAISS + BM25) ---
1. [Azure CycleCloud Documentation] Read in English Edit Azure CycleCloud Documentation Azure CycleCloud is designed to enable enterpris...
2. [Azure for developers overview] that are scalable, reliable, and maintainable. Azure supports the most popular programming languages...
3. [Azure DevOps documentation] Azure DevOps documentation Plan, build, test, and deploy with integrated DevOps tools for teams of a...




QUERY: How do I manage relation